# Temporal Calibration Pilot — DeepSeek-R1-Distill (configurable)

**Hypothesis:** a reasoning-mode LLM becomes *overconfident* on factual questions as the questioned events approach its training cutoff (≈ July 2024), and *Thinking ON* is worse-calibrated near the cutoff than *Thinking OFF*.

The 7B distill proved too weak factually (uniform ~8–22% accuracy floor, no temporal gradient). **Default model is now DeepSeek-R1-Distill-Qwen-14B in 4-bit** (same R1 family, so ON-vs-OFF stays a within-model contrast; ~10 GB, fits one 16 GB GPU). Override with no code edits via env vars: `TCP_MODEL_NAME`, `TCP_LOAD_IN_4BIT`, `TCP_MAX_NEW_TOKENS_ON`.

50 hand-curated questions × 2 modes = 100 calls. Click **Run All**. Outputs persist in `/kaggle/working/outputs/`, namespaced per model so switching models does not collide or false-resume.

## 1. Setup — packages, GPU check, locate the repo

In [ ]:
# Pull the LATEST code from GitHub. This is why a re-run never needs a Kaggle
# dataset upload again -- every run clones fresh `main`. The locator cell two
# down finds it under /kaggle/working (and is preferred over any stale dataset
# still attached under /kaggle/input).
import subprocess, shutil
from pathlib import Path

REPO_URL = "https://github.com/kumarswamyg2005/temporal-calib-pilot.git"
CLONE_DIR = Path("/kaggle/working/temporal-calib-pilot")

if CLONE_DIR.exists():
    shutil.rmtree(CLONE_DIR)
subprocess.run(
    ["git", "clone", "--depth", "1", "-b", "main", REPO_URL, str(CLONE_DIR)],
    check=True,
)
_head = subprocess.run(
    ["git", "-C", str(CLONE_DIR), "log", "-1", "--oneline"],
    capture_output=True, text=True,
).stdout.strip()
print("Cloned latest:", _head)

In [ ]:
# Install ONLY the model stack. We deliberately do NOT touch torch / numpy /
# pandas: Kaggle & Colab ship CUDA-matched builds and reinstalling them can
# break GPU inference. (requirements.txt has the full pinned set for local
# reproduction.)
#
# bitsandbytes enables 4-bit (nf4) loading so the 14B/32B distills fit a
# 16 GB GPU. NOTE: the full fp16 weights still download before quantization
# (14B ~28 GB) -- the FIRST run is download-bound; budget ~15-25 min for it.
#
# pip may print "ERROR: pip's dependency resolver ..." about a pre-installed
# Kaggle package (e.g. google-adk). That is harmless -- ignore it.
import sys, subprocess

PKGS = [
    "transformers==4.48.3",
    "accelerate==1.3.0",
    "bitsandbytes==0.45.0",
    "rapidfuzz==3.10.1",
    "seaborn==0.13.2",
    "python-dotenv==1.0.1",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *PKGS],
    check=True,
)
print("Model-stack packages installed.")

In [ ]:
# GPU check
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}  |  VRAM: {p.total_memory/1e9:.1f} GB")
else:
    print("WARNING: no GPU detected. Enable a GPU accelerator in Settings, "
          "or expect this to be extremely slow.")

In [ ]:
# Locate the repo root and put it on sys.path. Hardened for a dirty kernel:
#  - prefers the fresh /kaggle/working GitHub clone over any STALE attached
#    /kaggle/input dataset,
#  - drops other project paths from sys.path so they can't shadow it,
#  - evicts any already-imported `src` modules so a re-run in the SAME kernel
#    rebinds to the fresh code (this is the cause of a persistent
#    "cannot import name ... from src.config" after a failed earlier run).
import sys
from pathlib import Path

CLONE_DIR = Path("/kaggle/working/temporal-calib-pilot")
SEARCH_ROOTS = [CLONE_DIR, Path.cwd(), Path("/kaggle/working"), Path("/kaggle/input")]


def _find_repo():
    for root in SEARCH_ROOTS:
        if not root or not root.exists():
            continue
        if (root / "src" / "__init__.py").is_file():
            return root
        for hit in sorted(root.rglob("src/__init__.py")):
            return hit.parent.parent
    return None


REPO_ROOT = _find_repo()

if REPO_ROOT is None:
    print("Could not find src/__init__.py. What IS available:\n")
    for root in (Path("/kaggle/working"), Path("/kaggle/input"), Path.cwd()):
        print(f"--- {root}  (exists={root.exists()}) ---")
        if root.exists():
            for p in sorted(root.rglob("*"))[:40]:
                print("  ", p)
    raise FileNotFoundError(
        "Project files not found. Run the GitHub-clone cell above first "
        "(cell 2). If you imported an OLD copy of this notebook without that "
        "cell, re-import it from "
        "https://github.com/kumarswamyg2005/temporal-calib-pilot and use "
        "'Restart & Run All'."
    )

REPO_ROOT = REPO_ROOT.resolve()

# If we resolved to the stale dataset but a fresh clone exists, the clone
# should have won -- warn loudly so it is obvious.
if "/kaggle/input" in str(REPO_ROOT) and CLONE_DIR.exists():
    print("WARNING: resolved to the attached dataset despite a clone existing.")

# Remove any other entries that point at a copy of THIS project, then prepend.
sys.path[:] = [
    p for p in sys.path
    if p and Path(p).resolve() != REPO_ROOT
    and not (Path(p) / "src" / "config.py").is_file()
]
sys.path.insert(0, str(REPO_ROOT))

# Evict cached src.* so the next import binds to REPO_ROOT, not a stale path.
for _name in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_name]

print("Repo root:", REPO_ROOT)

## 2. Imports

In [ ]:
import time
import pandas as pd
from tqdm.auto import tqdm

from src.config import log, get_results_path, get_outputs_dir, slugify_model
from src.question_loader import load_questions, sanity_summary
from src.model_runner import (
    load_model, run_inference, cuda_memory_summary, MODEL_NAME, LOAD_IN_4BIT,
)
from src.evaluator import is_correct
from src.metrics import compute_summary, overall_by_mode
from src.visualizer import plot_overconfidence

MODES = ["thinking_on", "thinking_off"]
OUTPUTS = get_outputs_dir()
MODEL_SLUG = slugify_model(MODEL_NAME)
MODEL_LABEL = MODEL_NAME.split("/")[-1] + (" (4-bit)" if LOAD_IN_4BIT else " (fp16)")
RESULTS_CSV = get_results_path(MODEL_NAME)  # per-model -> no false-resume

print("Model       ->", MODEL_LABEL)
print("Outputs     ->", OUTPUTS)
print("Results CSV  ->", RESULTS_CSV.name)

## 3. Load questions + sanity check

In [ ]:
questions = load_questions()
print(sanity_summary(questions))
assert len(questions) == 50, "Expected exactly 50 pilot questions."

## 4. Load model (once) + memory usage

In [ ]:
_t0 = time.monotonic()
model, tokenizer = load_model()
print(cuda_memory_summary())
print(f"Model load took {time.monotonic() - _t0:.0f}s")

## 5. Run evaluation — 50 questions × 2 modes

Progress is logged with timestamps and checkpointed to `results.csv` every 10
questions, so an interrupted Kaggle kernel can resume (already-done
`(id, mode)` pairs are skipped on re-run).

In [ ]:
# Resume support: load any prior checkpoint for THIS model only.
if RESULTS_CSV.exists():
    results_df = pd.read_csv(RESULTS_CSV)
    if "model" in results_df.columns:
        results_df = results_df[results_df["model"] == MODEL_NAME]
    done = set(zip(results_df["id"], results_df["mode"]))
    rows = results_df.to_dict("records")
    log(f"Resuming: {len(done)} (id, mode) pairs already done for this model.")
else:
    rows, done = [], set()


def _checkpoint():
    pd.DataFrame(rows).to_csv(RESULTS_CSV, index=False)


eval_start = time.monotonic()
total = len(questions)
for qi, q in enumerate(tqdm(questions, desc="questions"), start=1):
    for mode in MODES:
        if (q.id, mode) in done:
            continue
        log(f"Running question {qi}/{total} ({q.bucket}, {mode}) [{q.id}]")
        out = run_inference(q.question, mode, model, tokenizer)
        correct = is_correct(out["answer"], q.answer, q.answer_aliases)
        rows.append({
            "id": q.id,
            "model": MODEL_NAME,
            "bucket": q.bucket,
            "distance_months": q.distance_months,
            "category": q.category,
            "mode": mode,
            "question": q.question,
            "true_answer": q.answer,
            "predicted_answer": out["answer"],
            "confidence": out["confidence"],
            "is_correct": bool(correct),
            "parse_ok": out["parse_ok"],
            "tokens_used": out["tokens_used"],
            "reasoning_chars": out["reasoning_chars"],
            "full_response": out["full_response"],
        })
    if qi % 10 == 0:
        _checkpoint()
        log(f"Checkpoint saved at question {qi} ({len(rows)} rows).")

_checkpoint()
eval_minutes = (time.monotonic() - eval_start) / 60.0
results_df = pd.DataFrame(rows)
log(f"Evaluation complete: {len(results_df)} rows in {eval_minutes:.1f} min.")
assert len(results_df) == 100, f"Expected 100 rows, got {len(results_df)}."

## 6. Compute metrics

In [ ]:
summary = compute_summary(results_df)
summary_path = OUTPUTS / f"summary_table__{MODEL_SLUG}.csv"
summary.to_csv(summary_path, index=False)

print("Per-bucket / per-mode summary:\n")
print(summary.to_string(index=False))
print("\nPooled by mode:\n")
print(overall_by_mode(results_df).to_string(index=False))
print("\nSaved ->", summary_path)

## 7. Headline chart

In [ ]:
from IPython.display import Image, display

chart_path = OUTPUTS / f"pilot_chart__{MODEL_SLUG}.png"
plot_overconfidence(summary, chart_path, model_label=MODEL_LABEL)
display(Image(filename=str(chart_path)))
print("Saved ->", chart_path)

## 8. Final summary + verdict

In [ ]:
def _gap(bucket, mode):
    r = summary[(summary["bucket"] == bucket) & (summary["mode"] == mode)]
    return float(r["overconfidence_gap"].iloc[0]) if len(r) else float("nan")

by_mode = overall_by_mode(results_df).set_index("mode")
acc_on = by_mode.loc["thinking_on", "accuracy"] * 100
acc_off = by_mode.loc["thinking_off", "accuracy"] * 100

g_on_b5, g_off_b5 = _gap("B5", "thinking_on"), _gap("B5", "thinking_off")
g_on_b1, g_off_b1 = _gap("B1", "thinking_on"), _gap("B1", "thinking_off")

# Signal = (ON more overconfident than OFF near the cutoff) AND
#          (ON more overconfident near the cutoff than far from it).
mode_diff_near = g_on_b5 - g_off_b5
on_drift = g_on_b5 - g_on_b1
strength = min(mode_diff_near, on_drift)

if strength >= 15:
    verdict = ("Thinking ON shows clearly worse calibration near cutoff. "
               "Strong signal — scale up.")
elif strength >= 5:
    verdict = ("Thinking ON shows weakly worse calibration near cutoff. "
               "Mild signal — scale up to confirm.")
else:
    verdict = ("No clear difference between modes. "
               "Pivot or rethink the hypothesis.")

# Floor-effect guard: if accuracy is near zero everywhere the model is simply
# too weak and a 'gap' is uninformative (the 7B failure mode). Flag it.
floor_warn = ""
if acc_on < 30 and acc_off < 30:
    floor_warn = ("\nWARNING: accuracy < 30% in both modes -- likely a floor "
                  "effect; the temporal gradient cannot be read reliably. "
                  "Use a stronger model.")

print("PILOT STUDY COMPLETE")
print("====================")
print(f"Model: {MODEL_LABEL}")
print(f"Total questions evaluated: {len(results_df)}")
print(f"Total inference time: {eval_minutes:.1f} minutes")
print(f"Thinking ON accuracy: {acc_on:.1f}%")
print(f"Thinking OFF accuracy: {acc_off:.1f}%")
print(f"Overconfidence gap at B5 (1mo): ON={g_on_b5:.1f}, OFF={g_off_b5:.1f}")
print(f"Overconfidence gap at B1 (24mo): ON={g_on_b1:.1f}, OFF={g_off_b1:.1f}")
print()
print(f"VERDICT: {verdict}{floor_warn}")
print()
print(f"Chart saved to: {chart_path}")